# 任务 1：亿级数据精密去重

## 任务描述
基于 `user_id`, `item_id`, `behavior_type`, `timestamp` 四个维度进行数据去重

## 技术栈
- Polars (Lazy 模式)
- Parquet 格式

In [ ]:
import polars as pl

print(f"Polars 版本：{pl.__version__}")

## 1. 加载数据（Lazy 模式）

In [ ]:
# 加载清洗后的数据
q = pl.scan_parquet("clean_data_partitioned/**/*.parquet")
print("数据加载完成（LazyFrame 模式）")

## 2. 去重前后统计

In [ ]:
# 获取去重前行数
initial_count = q.collect().height
print(f"去重前行数：{initial_count:,}")

## 3. 精密去重逻辑

使用 Polars 的 `.unique()` 方法，基于四个关键字段去重

In [ ]:
# 精密去重：基于 user_id, item_id, behavior_type, timestamp
q_unique = q.unique(
    subset=["user_id", "item_id", "behavior_type", "timestamp"],
    keep="first"  # 保留首次出现的记录
)

# 获取去重后行数
final_count = q_unique.collect().height
print(f"去重后行数：{final_count:,}")

## 4. 结果统计

In [ ]:
# 计算去重统计
removed_count = initial_count - final_count
dedup_ratio = (1 - final_count / initial_count) * 100 if initial_count > 0 else 0

print("=" * 60)
print("去重前后对比")
print("=" * 60)
print(f"去重前行数：{initial_count:,}")
print(f"去重后行数：{final_count:,}")
print(f"移除冗余：{removed_count:,} 条")
print(f"去重比例：{dedup_ratio:.2f}%")

## 5. 保存去重结果

In [ ]:
# 保存去重后的数据
q_unique.sink_parquet("clean_data_deduped.parquet")
print("去重数据已保存至：clean_data_deduped.parquet")